In [4]:
from pathlib import Path
import pandas as pd

RAW_DATA_DIR = Path("data/raw")
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:")
for file in csv_files:
    print(f"- {file.name}")

Found 9 CSV files:
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


# Olist E-Commerce Data Audit

This notebook inventories the raw datasets and identifies potential data-quality issues before cleaning, transformation, and analysis.

## Objectives

- Confirm that all required datasets are present
- Measure each dataset's size
- Identify missing values and duplicate records
- Document issues requiring cleaning

In [5]:
datasets = {}

for file in csv_files:
    dataset_name = file.stem
    datasets[dataset_name] = pd.read_csv(file, low_memory=False)

    rows, columns = datasets[dataset_name].shape
    print(f"{dataset_name}: {rows:,} rows × {columns} columns")

olist_customers_dataset: 99,441 rows × 5 columns
olist_geolocation_dataset: 1,000,163 rows × 5 columns
olist_order_items_dataset: 112,650 rows × 7 columns
olist_order_payments_dataset: 103,886 rows × 5 columns
olist_order_reviews_dataset: 99,224 rows × 7 columns
olist_orders_dataset: 99,441 rows × 8 columns
olist_products_dataset: 32,951 rows × 9 columns
olist_sellers_dataset: 3,095 rows × 4 columns
product_category_name_translation: 71 rows × 2 columns


In [6]:
audit_records = []

for name, df in datasets.items():
    audit_records.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

audit_summary = (
    pd.DataFrame(audit_records)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

audit_summary

,dataset,rows,columns,missing_values,duplicate_rows
0,olist_geolocation_dataset,1000163,5,0,261831
1,olist_order_items_dataset,112650,7,0,0
2,olist_order_payments_dataset,103886,5,0,0
3,olist_customers_dataset,99441,5,0,0
4,olist_orders_dataset,99441,8,4908,0
5,olist_order_reviews_dataset,99224,7,145903,0
6,olist_products_dataset,32951,9,2448,0
7,olist_sellers_dataset,3095,4,0,0
8,product_category_name_translation,71,2,0,0


In [7]:
missing_records = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        if missing_count > 0:
            missing_records.append({
                "dataset": name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_by_column = (
    pd.DataFrame(missing_records)
    .sort_values(
        ["missing_percentage", "missing_count"],
        ascending=False
    )
    .reset_index(drop=True)
)

missing_by_column

,dataset,column,missing_count,missing_percentage
0,olist_order_reviews_dataset,review_comment_title,87656,88.34
1,olist_order_reviews_dataset,review_comment_message,58247,58.70
2,olist_orders_dataset,order_delivered_customer_date,2965,2.98
3,olist_products_dataset,product_category_name,610,1.85
4,olist_products_dataset,product_name_lenght,610,1.85
5,olist_products_dataset,product_description_lenght,610,1.85
6,olist_products_dataset,product_photos_qty,610,1.85
7,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
8,olist_orders_dataset,order_approved_at,160,0.16
9,olist_products_dataset,product_weight_g,2,0.01


In [8]:
orders = datasets["olist_orders_dataset"]

order_status_audit = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_approval_dates=(
            "order_approved_at",
            lambda values: int(values.isna().sum())
        ),
        missing_carrier_dates=(
            "order_delivered_carrier_date",
            lambda values: int(values.isna().sum())
        ),
        missing_customer_delivery_dates=(
            "order_delivered_customer_date",
            lambda values: int(values.isna().sum())
        )
    )
    .sort_values("total_orders", ascending=False)
)

order_status_audit

,total_orders,missing_approval_dates,missing_carrier_dates,missing_customer_delivery_dates
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [9]:
key_definitions = {
    "olist_customers_dataset": ["customer_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id"],
    "product_category_name_translation": ["product_category_name"]
}

key_audit_records = []

for dataset_name, key_columns in key_definitions.items():
    df = datasets[dataset_name]
    key_data = df[key_columns]

    key_audit_records.append({
        "dataset": dataset_name,
        "proposed_key": " + ".join(key_columns),
        "total_rows": len(df),
        "null_key_rows": int(key_data.isna().any(axis=1).sum()),
        "rows_in_duplicate_keys": int(
            key_data.duplicated(keep=False).sum()
        ),
        "key_is_unique": not key_data.duplicated().any()
    })

key_audit = pd.DataFrame(key_audit_records)
key_audit

,dataset,proposed_key,total_rows,null_key_rows,rows_in_duplicate_keys,key_is_unique
0,olist_customers_dataset,customer_id,99441,0,0,True
1,olist_orders_dataset,order_id,99441,0,0,True
2,olist_products_dataset,product_id,32951,0,0,True
3,olist_sellers_dataset,seller_id,3095,0,0,True
4,olist_order_items_dataset,order_id + order_item_id,112650,0,0,True
5,olist_order_payments_dataset,order_id + payment_sequential,103886,0,0,True
6,olist_order_reviews_dataset,review_id,99224,0,1603,False
7,product_category_name_translation,product_category_name,71,0,0,True


In [10]:
reviews = datasets["olist_order_reviews_dataset"]

review_key_candidates = {
    "review_id": ["review_id"],
    "order_id": ["order_id"],
    "review_id + order_id": ["review_id", "order_id"],
    "review_id + order_id + creation_date": [
        "review_id",
        "order_id",
        "review_creation_date"
    ]
}

review_key_results = []

for key_name, columns in review_key_candidates.items():
    key_data = reviews[columns]

    review_key_results.append({
        "candidate_key": key_name,
        "rows_in_duplicate_keys": int(
            key_data.duplicated(keep=False).sum()
        ),
        "duplicate_records_beyond_first": int(
            key_data.duplicated().sum()
        ),
        "is_unique": not key_data.duplicated().any()
    })

pd.DataFrame(review_key_results)

,candidate_key,rows_in_duplicate_keys,duplicate_records_beyond_first,is_unique
0,review_id,1603,814,False
1,order_id,1098,551,False
2,review_id + order_id,0,0,True
3,review_id + order_id + creation_date,0,0,True


In [11]:
duplicate_review_ids = (
    reviews[
        reviews["review_id"].duplicated(keep=False)
    ]
    .sort_values(["review_id", "order_id"])
)

duplicate_review_ids.head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [12]:
relationships = [
    (
        "orders → customers",
        "olist_orders_dataset",
        "customer_id",
        "olist_customers_dataset",
        "customer_id"
    ),
    (
        "items → orders",
        "olist_order_items_dataset",
        "order_id",
        "olist_orders_dataset",
        "order_id"
    ),
    (
        "items → products",
        "olist_order_items_dataset",
        "product_id",
        "olist_products_dataset",
        "product_id"
    ),
    (
        "items → sellers",
        "olist_order_items_dataset",
        "seller_id",
        "olist_sellers_dataset",
        "seller_id"
    ),
    (
        "payments → orders",
        "olist_order_payments_dataset",
        "order_id",
        "olist_orders_dataset",
        "order_id"
    ),
    (
        "reviews → orders",
        "olist_order_reviews_dataset",
        "order_id",
        "olist_orders_dataset",
        "order_id"
    ),
    (
        "products → category translations",
        "olist_products_dataset",
        "product_category_name",
        "product_category_name_translation",
        "product_category_name"
    )
]

relationship_results = []

for (
    relationship_name,
    child_name,
    child_column,
    parent_name,
    parent_column
) in relationships:

    child_values = datasets[child_name][child_column].dropna()
    parent_values = set(
        datasets[parent_name][parent_column].dropna()
    )

    orphan_mask = ~child_values.isin(parent_values)

    relationship_results.append({
        "relationship": relationship_name,
        "non_null_child_rows": len(child_values),
        "orphan_rows": int(orphan_mask.sum()),
        "orphan_unique_values": int(
            child_values[orphan_mask].nunique()
        ),
        "relationship_valid": not orphan_mask.any()
    })

relationship_audit = pd.DataFrame(relationship_results)
relationship_audit

,relationship,non_null_child_rows,orphan_rows,orphan_unique_values,relationship_valid
0,orders → customers,99441,0,0,True
1,items → orders,112650,0,0,True
2,items → products,112650,0,0,True
3,items → sellers,112650,0,0,True
4,payments → orders,103886,0,0,True
5,reviews → orders,99224,0,0,True
6,products → category translations,32341,13,2,False


In [13]:
products = datasets["olist_products_dataset"]
translations = datasets["product_category_name_translation"]

known_categories = set(
    translations["product_category_name"].dropna()
)

untranslated_products = products[
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(known_categories)
]

untranslated_category_summary = (
    untranslated_products
    .groupby("product_category_name")
    .agg(
        product_count=("product_id", "nunique")
    )
    .reset_index()
    .sort_values("product_count", ascending=False)
)

untranslated_category_summary

,product_category_name,product_count
1,portateis_cozinha_e_preparadores_de_alimentos,10
0,pc_gamer,3


In [14]:
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

audit_summary.to_csv(
    REPORTS_DIR / "dataset_inventory.csv",
    index=False
)

missing_by_column.to_csv(
    REPORTS_DIR / "missing_values_audit.csv",
    index=False
)

key_audit.to_csv(
    REPORTS_DIR / "primary_key_audit.csv",
    index=False
)

relationship_audit.to_csv(
    REPORTS_DIR / "relationship_audit.csv",
    index=False
)

untranslated_category_summary.to_csv(
    REPORTS_DIR / "untranslated_categories.csv",
    index=False
)

print("Audit reports saved:")
for file in sorted(REPORTS_DIR.glob("*.csv")):
    print(f"- {file.name}")

Audit reports saved:
- dataset_inventory.csv
- missing_values_audit.csv
- primary_key_audit.csv
- relationship_audit.csv
- untranslated_categories.csv


## Initial Audit Conclusions

- All nine expected datasets were successfully located and loaded.
- Most transactional tables contain valid unique or compound keys.
- The reviews table requires `review_id + order_id` as its compound key.
- Missing review titles and messages represent optional customer input and should be retained.
- Most missing order dates reflect normal order lifecycle states such as shipped, cancelled, unavailable, or processing.
- A small number of delivered orders contain missing milestone dates and should be flagged for analysis.
- The geolocation table contains exact duplicate rows that require removal before geographic aggregation.
- Thirteen products use two legitimate categories missing from the English translation reference table; controlled translations will be added.
- Raw source files will remain unchanged throughout the project.